## Building a RAG-System with Langchain and ChromaDB
### Introduction

Retrieval Augumented Generation(RAG) is a powerful technique that contains the capabilities of LLM with external knowledge retrival. This notebook will walk you through building a complete RAG system using:
- Langchain: A framework for developing applications by language models
- ChromeDB: An open-source vector DB for storing and retrieving embeddings
- OpenAI: for mebddings and language model (you can substitute with other providers)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
## Langchain Imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document


## Vector Stores
from langchain_community.vectorstores import Chroma

## Utility Imports
import numpy as np
import os
from typing import List

f:\D Drive\Udemy_RAG (Krish Naik)\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hp\AppData\Local\Temp\ipykernel_34332\1481314563.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


### 1. Sample Data

In [3]:
## Create Sample Documents
sample_docs = [
    """
    Machine Learning fundamentals

    Machine Learning fundamentals focus on teaching computers to learn patterns from data
    without being explicitly programmed for every task. The main types of machine learning
    are supervised learning, unsupervised learning, and reinforcement learning. In supervised
    learning, models learn from labeled data to make predictions or classifications. Common
    algorithms include Linear Regression, Logistic Regression, Decision Trees, Random Forests,
    Support Vector Machines, K-Nearest Neighbors, and Gradient Boosting. Unsupervised learning
    works with unlabeled data and is commonly used for clustering and dimensionality reduction.
    Important concepts include training and testing datasets, feature engineering, model
    evaluation, overfitting, underfitting, bias, variance, and cross-validation. Common
    evaluation metrics include accuracy, precision, recall, F1-score, MAE, and RMSE.
    """

    ,
    """
    Deep Learning and Neural Networks

    Deep Learning is a subfield of machine learning that uses artificial neural networks
    with multiple layers to learn complex patterns from large amounts of data. A neural
    network consists of interconnected neurons organized into input, hidden, and output layers.
    Each neuron applies weights, a bias, and an activation function to its inputs. Popular
    activation functions include ReLU, Sigmoid, and Tanh. During training, the network
    calculates a loss and uses backpropagation to determine how the weights should be updated.
    Optimization algorithms such as Gradient Descent and Adam are commonly used for training.
    Convolutional Neural Networks are widely used for computer vision, while Recurrent Neural
    Networks, LSTMs, and GRUs were traditionally used for sequential data. Modern deep learning
    also heavily relies on Transformer architectures for language, vision, and multimodal tasks.
    """

    ,
    """
    Natural Language Processing (NLP) 
    
    Natural Language Processing, or NLP, is a field of artificial intelligence that focuses
    on enabling computers to understand, process, and generate human language. NLP techniques
    are used in applications such as chatbots, machine translation, sentiment analysis,
    text classification, information extraction, and question answering. Text preprocessing
    may include tokenization, normalization, stop-word removal, stemming, and lemmatization.
    Traditional NLP systems often represented text using methods such as Bag of Words and TF-IDF.
    Word embeddings such as Word2Vec, GloVe, and FastText introduced dense numerical
    representations of words. Modern NLP uses Transformer-based models such as BERT, T5,
    and GPT-style models. Attention mechanisms allow models to focus on relevant parts of
    an input sequence. Large Language Models can perform tasks such as summarization,
    translation, reasoning, question answering, and text generation.
    """
]

In [4]:
## Save sample documents to files
import tempfile
temp_dir = tempfile.mkdtemp() ## Creating a temperorary directory

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample document create in: {temp_dir}")

Sample document create in: C:\Users\hp\AppData\Local\Temp\tmpjgdiexhx


In [5]:
## Save sample documents to files
import tempfile
temp_dir = tempfile.mkdtemp() ## Creating a temperorary directory

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)

### 2. Document Loading

In [6]:
from langchain_community.document_loaders import DirectoryLoader

## Load Directories from the directory
loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\n First Document preview.")
print(documents[0].page_content[:200] + "......")

Loaded 3 documents

 First Document preview.

    Machine Learning fundamentals

    Machine Learning fundamentals focus on teaching computers to learn patterns from data
    without being explicitly programmed for every task. The main types of ......


In [7]:
documents

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machine Learning fundamentals\n\n    Machine Learning fundamentals focus on teaching computers to learn patterns from data\n    without being explicitly programmed for every task. The main types of machine learning\n    are supervised learning, unsupervised learning, and reinforcement learning. In supervised\n    learning, models learn from labeled data to make predictions or classifications. Common\n    algorithms include Linear Regression, Logistic Regression, Decision Trees, Random Forests,\n    Support Vector Machines, K-Nearest Neighbors, and Gradient Boosting. Unsupervised learning\n    works with unlabeled data and is commonly used for clustering and dimensionality reduction.\n    Important concepts include training and testing datasets, feature engineering, model\n    evaluation, overfitting, underfitting, bias, variance, and cross-validation. Common\n    evaluation metrics include accuracy, precision, recall

### Document Splitting

In [8]:
## Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=500, # Maximum size of each chunk
    chunk_overlap = 50, ## Overlap between chunks to maintain content
    length_function = len,
    separators=["\n\n","\n","."," ",""] # hierarchy of separotrs
)
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print("\n Chunk Example:")
print(f"Content: {chunks[0].page_content[:150]} ...")
print(f"Metadata: {chunks[0].metadata}")

Created 10 chunks from 3 documents

 Chunk Example:
Content: Machine Learning fundamentals ...
Metadata: {'source': 'data\\doc_0.txt'}


### Embedding Models

In [ ]:
os.eniron["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
sample_text= "Machine Learning is fascinating."
embeddings = OpenAIEmbeddings()
embeddings

In [ ]:
vector = embeddings.embed_query(sample_text)

### initialize the ChromaBD Vector Store DB and stores the chunks in Vector Representation

In [ ]:
## Create a chromabd vector store
persistant_directory = './chroma_db'

## initialize chromadb with openai embeddings
vector_store = Chroma.from_documents(
    documents=chunks,
    embeddings = OpenAIEmbeddings(),
    persist_directory=persistant_directory,
    collection_name='rag_collection'
)

print(f"Vector Store created with: {vector_store._collection.count()} vectors")
print(f"Persisted to: {persistant_directory}")

### Test the Similarity Search

In [ ]:
query = "What are the types of Machine Learning?"

similar_docs = vector_store.similarity_search(query,k = 3)
similar_docs

In [ ]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i,doc in enumerate(similar_docs):
    print(f"\n --- Chunk{i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source','Unknown')}")

### Advanced Similarity search with scores

In [ ]:
result_scores = vector_store.similarity_search_with_relevance_scores(query,k = 3)
print(result_scores)

#### 2. understanding Similarity scores
The similarity score represents now closely related a document related a document chunk is to your query. The scoring depends on the distance metric used.

ChromaDB default: Uses L2 distance( Eucllidean DIstance)
- Lower Scores = MORE similar (vloser in vector space)
- Score of 0 - Identical Vectors
- Typical range: 0 to 2 ( but can be higher)

Cosine Similarity ( if configured):
- Higher Scores = MORE similar
- Range: -1 to 1 ( 1 being identical )